# **Telco Customer Churn Dataset**
**Step 1: Problem Understanding**

The objective of this project is to predict whether a telecom customer is likely to churn, i.e., stop using the company’s services. Churn prediction helps businesses identify at-risk customers and take proactive actions such as offering discounts or improving service quality.

This is a binary classification problem, where:

*   1 → Customer will churn
*   0 → Customer will not churn





In [79]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


**Step 2:  Dataset Description**

The project uses the Telco Customer Churn dataset, which contains customer demographic details, account information, service usage, and billing data.

Each row represents a customer.

The dataset includes numerical and categorical features.

The target variable is Churn.

**Step 3️ : Data Loading**

The dataset is loaded using Pandas. Initial inspection is performed to understand:

* Number of rows and columns

* Data types

* Presence of missing values

This step ensures familiarity with the structure of the dataset before preprocessing.

In [80]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


**Step 4 : Data Cleaning**


*   Data cleaning is performed to make the dataset suitable for modeling:

*   The TotalCharges column is converted from string to numeric.

* Invalid or blank values are converted to missing values (NaN).

* The customerID column is removed because it does not contribute to prediction.

* Missing values are handled later using imputation in the pipeline.

This step improves data quality and model reliability.

In [81]:
df.isna().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [82]:
df.drop('customerID', axis = 1, inplace=True)

In [13]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


**Step 5 :  Feature and Target Separation**

The dataset is split into:

* Input features (X): Customer attributes such as tenure, contract type, payment method, etc.

* Target variable (y): Churn, converted into binary values (Yes → 1, No → 0).

Separating features and target is essential for supervised learning.

In [83]:
x = df.drop('Churn', axis = 1)
y = df['Churn'].map({'Yes' : 1, 'No' : 0})

In [84]:
x

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6


In [85]:
y

,Churn
0,0
1,0
2,1
3,0
4,1
...,...
7038,0
7039,0
7040,0
7041,1


**Step 6️ : Feature Type Identification**
Features are divided into:

* Numerical features: tenure, MonthlyCharges, TotalCharges

* Categorical features: Gender, contract type, internet service, payment method, etc.

This separation allows appropriate preprocessing techniques to be applied to each feature type.

In [86]:
num_features = x.select_dtypes(include = ['int64', 'float64']).columns
num_features

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')

In [87]:
cate_features = x.select_dtypes(include = ['object']).columns
cate_features

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object')

**Step 7️ :  Data Preprocessing**

A preprocessing pipeline is created to handle data transformation consistently:

* Numerical features

* Missing values are filled using the median.

* Features are scaled using standardization.

* Categorical features

* Missing values are filled using the most frequent value.

* One-Hot Encoding is applied to convert categories into numerical form.

This step ensures all features are in a model-ready format.

In [66]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])


**Step 8 : Pipeline Construction**

A machine learning pipeline is built that combines:

* Data preprocessing

* Model training

Using a pipeline ensures:

* No data leakage

* Same preprocessing is applied during training and testing

* Clean and maintainable code

In [69]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cate_features)
    ])

**Step 9️ : Train–Test Split**

The dataset is split into:

* Training set (80%) – used to train the model

*Testing set (20%) – used to evaluate performance

Stratified sampling is used to maintain the churn distribution in both sets.

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify = y
)

**Step 10 : Model Selection**

Multiple machine learning models are trained to improve accuracy and compare performance:

1) Logistic Regression (baseline)

2) K-Nearest Neighbors

3) Decision Tree

4) Random Forest

5) Gradient Boosting

Each model is given a unique identifier for clarity and comparison.

In [88]:
models = {
    "LR-ChurnNet": LogisticRegression(max_iter=1000),
    "KNN-ChurnScope": KNeighborsClassifier(n_neighbors=7),
    "DT-ChurnTree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "RF-ChurnForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GB-ChurnBoost": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05)
}


**Step 11 : Model Training and Testing  with Model Evaluation**
Models are evaluated using metrics suitable for churn prediction:

* Accuracy – Overall correctness

* Precision – How many predicted churners actually churned

* Recall – How many actual churners were correctly identified

* F1-Score – Balance between precision and recall

* ROC-AUC – Ability to distinguish between churn and non-churn

F1-Score and ROC-AUC are prioritized due to class imbalance.

In [89]:
results = []

for model_name, classifier in models.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })


**Step 14 : Model Comparison**

All model results are stored in a comparison table.
Models are ranked based on F1-Score and ROC-AUC to identify the best-performing algorithm.

This helps select the most effective model for deployment.

In [90]:
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print(results_df)


            Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
0     LR-ChurnNet  0.805536   0.657233  0.558824  0.604046  0.842008
4   GB-ChurnBoost  0.802697   0.664384  0.518717  0.582583  0.843525
3  RF-ChurnForest  0.790632   0.632997  0.502674  0.560358  0.826427
1  KNN-ChurnScope  0.767921   0.564384  0.550802  0.557510  0.802439
2    DT-ChurnTree  0.797019   0.674603  0.454545  0.543131  0.827764


The project demonstrates an end-to-end machine learning workflow for churn prediction. Ensemble models such as **Random Forest** and **Gradient Boosting** typically outperform baseline models due to their ability to capture complex customer behavior patterns.

**How to Improve Churn Prediction Performance**

In [95]:
from sklearn.ensemble import RandomForestClassifier

rf_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight='balanced',
        random_state=42
    ))
])

rf_balanced.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='f...
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        n_estimators=300, random_state=42))])

In [98]:

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

df['AvgCharges'] = df['TotalCharges'] / (df['tenure'] + 1)
df['HighMonthly'] = (df['MonthlyCharges'] > 70).astype(int)


/tmp/ipython-input-833647804.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [99]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='f...
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(max_depth=12, n_estimators=300,
                                        random_state=42))])

In [100]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'classifier__n_estimators': [200, 300],
    'classifier__max_depth': [8, 12],
    'classifier__min_samples_split': [5, 10]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    scoring='f1',
    cv=5
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best Parameters:", grid.best_params_)


Best Parameters: {'classifier__max_depth': 12, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}


In [101]:
y_prob = best_model.predict_proba(X_test)[:, 1]

# Custom threshold
y_pred_tuned = (y_prob > 0.35).astype(int)

from sklearn.metrics import f1_score, recall_score

print("F1 Score (Tuned):", f1_score(y_test, y_pred_tuned))
print("Recall (Tuned):", recall_score(y_test, y_pred_tuned))


F1 Score (Tuned): 0.6241299303944315
Recall (Tuned): 0.7192513368983957


In [102]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.89      0.79      0.83      1035
           1       0.55      0.72      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.77      0.78      1409

ROC-AUC: 0.8425637448655352
